# 🛡️ Phân Loại Mã Độc bằng CNN
**Kiến trúc:** Convolutional Neural Network (CNN)  
**Định dạng ảnh đầu vào:** 8×32×1 (grayscale)

---
### Hướng dẫn sử dụng
1. **Mount Google Drive** (Cell 2) để truy cập dataset
2. **Cấu hình tham số** tại Cell 3 (đường dẫn dataset, random seed)
3. Chạy tuần tự từ trên xuống dưới (`Runtime → Run all`)

> ⚠️ Khuyến nghị: bật **GPU** (`Runtime → Change runtime type → T4 GPU`) để tăng tốc huấn luyện.

## 📦 1. Cài Đặt Thư Viện

In [ ]:
# Cài thêm openpyxl nếu chưa có (các thư viện còn lại đã có sẵn trên Colab)
!pip install -q openpyxl

## ☁️ 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive đã được mount thành công!')

## ⚙️ 3. Cấu Hình Tham Số

In [ ]:
# ============================================================
# THAY ĐỔI CÁC GIÁ TRỊ NÀY TRƯỚC KHI CHẠY
# ============================================================

# Đường dẫn tới thư mục dataset trên Google Drive
# Ví dụ: '/content/drive/MyDrive/dataset/malware_images'
# Cấu trúc thư mục cần có:
#   dataset/
#     benign/   <- chứa các file .png benign
#     malware_family_1/  <- chứa .png mã độc
#     malware_family_2/
#     ...
DATASET_PATH = '/content/drive/MyDrive/dataset'  # <-- SỬA ĐƯỜNG DẪN NÀY

# Random seed để tái lặp kết quả
RANDOM_SEED = 42  # <-- SỬA NẾU CẦN

# Siêu tham số huấn luyện
BATCH_SIZE = 32
EPOCHS = 10
TEST_SPLIT = 0.3

# Thư mục lưu kết quả đầu ra
OUTPUT_DIR = '/content/drive/MyDrive/cnn_output'  # <-- SỬA NẾU CẦN

print(f'📂 Dataset path : {DATASET_PATH}')
print(f'🎲 Random seed  : {RANDOM_SEED}')
print(f'📊 Batch size   : {BATCH_SIZE}')
print(f'🔁 Epochs       : {EPOCHS}')
print(f'📁 Output dir   : {OUTPUT_DIR}')

## 📚 4. Import Thư Viện

In [ ]:
import os
import glob
import random
import time
import json
import numpy as np
import pandas as pd
from PIL import Image

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import openpyxl

# Tạo thư mục output
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'dat'), exist_ok=True)

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU available      : {len(tf.config.list_physical_devices("GPU")) > 0}')

## 📂 5. Đọc & Cân Bằng Dữ Liệu (Data Balancing)

In [ ]:
# Kiểm tra dataset tồn tại
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"❌ Không tìm thấy dataset tại: {DATASET_PATH}")

list_fams = os.listdir(DATASET_PATH)
benign_paths = []
malware_paths = []

# Bước 5.1: Gom đường dẫn file
for fam_name in list_fams:
    fam_dir = os.path.join(DATASET_PATH, fam_name)
    if not os.path.isdir(fam_dir):
        continue
    img_list = glob.glob(os.path.join(fam_dir, '*.png'))
    if fam_name.lower() == 'benign':
        benign_paths.extend(img_list)
    else:
        malware_paths.extend(img_list)

# Bước 5.2: Cân bằng dữ liệu (Malware = 2 × Benign)
num_benign = len(benign_paths)
target_malware = num_benign * 2

print(f'Tổng số ảnh Benign gốc  : {num_benign}')
print(f'Tổng số ảnh Malware gốc : {len(malware_paths)}')

if len(malware_paths) > target_malware:
    random.seed(RANDOM_SEED)
    malware_paths = random.sample(malware_paths, target_malware)

print(f'=> Số ảnh Malware sau Downsampling: {len(malware_paths)}')

In [ ]:
# Bước 5.3: Pipeline tiền xử lý & số hóa dữ liệu
X = []
y = []

print('Đang đọc ảnh Benign...')
for img_path in benign_paths:
    with Image.open(img_path) as im:
        im1 = im.resize((8, 32), Image.Resampling.LANCZOS)
        X.append(np.array(im1))
        y.append([1, 0])

print('Đang đọc ảnh Malware...')
for img_path in malware_paths:
    with Image.open(img_path) as im:
        im1 = im.resize((8, 32), Image.Resampling.LANCZOS)
        X.append(np.array(im1))
        y.append([0, 1])

# Chuẩn hóa về [0, 1]
X = np.array(X).astype('float32') / 255.0
y = np.array(y)
total_samples = len(X)
X = X.reshape(total_samples, 8, 32, 1)

print(f'\nTổng số mẫu : {total_samples}')
print(f'Shape X     : {X.shape}')
print(f'Shape y     : {y.shape}')

## ✂️ 6. Phân Chia Dữ Liệu Train / Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SPLIT, random_state=RANDOM_SEED
)

print('-' * 45)
print('THỐNG KÊ SỐ LƯỢNG ẢNH CỦA 2 LỚP')
print('-' * 45)
print(f'TỔNG CỘNG : Benign = {int(np.sum(y[:, 0]))} | Malware = {int(np.sum(y[:, 1]))}')
print(f'-> TRAIN  : Benign = {int(np.sum(y_train[:, 0]))} | Malware = {int(np.sum(y_train[:, 1]))}')
print(f'-> TEST   : Benign = {int(np.sum(y_test[:, 0]))} | Malware = {int(np.sum(y_test[:, 1]))}')
print('-' * 45)

## 🧠 7. Xây Dựng & Huấn Luyện Mô Hình CNN

In [ ]:
malware_model = Sequential([
    Conv2D(16, kernel_size=(3, 3), strides=(1, 1), padding='valid',
           activation='relu', input_shape=(8, 32, 1), name='Conv2D_1'),
    MaxPooling2D(pool_size=(2, 2), name='MaxPooling_1'),
    Conv2D(32, kernel_size=(3, 3), strides=(1, 1), padding='valid',
           activation='relu', name='Conv2D_2'),
    Flatten(name='Flatten_Core'),
    Dense(48, activation='relu', name='Dense_1'),
    Dense(2, activation='softmax', name='Dense_2')
])

malware_model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

malware_model.summary()

In [ ]:
print('Bắt đầu huấn luyện mô hình...')
tic = time.time()

history = malware_model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    verbose=1,
    validation_split=0.2
)

toc = time.time()
print(f'\n✅ Thời gian huấn luyện: {toc - tic:.4f} giây')

# Lưu mô hình
model_path = os.path.join(OUTPUT_DIR, 'cnn_bin_balanced_hardware_model.h5')
malware_model.save(model_path)
print(f'💾 Mô hình đã lưu tại: {model_path}')

## 📈 8. Biểu Đồ Quá Trình Huấn Luyện

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], label='Train Accuracy')
ax1.plot(history.history['val_accuracy'], label='Val Accuracy')
ax1.set_title('Accuracy qua từng Epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(history.history['loss'], label='Train Loss')
ax2.plot(history.history['val_loss'], label='Val Loss')
ax2.set_title('Loss qua từng Epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, 'training_history.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'📊 Biểu đồ lưu tại: {plot_path}')

## 🔍 9. Đánh Giá Mô Hình

In [ ]:
tic = time.time()
y_predict = malware_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
toc = time.time()
print(f'⏱️  Thời gian suy luận: {toc - tic:.4f} giây')

test_eval = malware_model.evaluate(X_test, y_test, verbose=0)
print(f'\n📌 Test Accuracy : {test_eval[1]:.4f}')
print(f'📌 Test Loss     : {test_eval[0]:.4f}')

y_true_classes = y_test.argmax(axis=1)
y_pred_classes = y_predict.argmax(axis=1)
target_names = ['Benign', 'Malware']

print('\n' + '=' * 50)
print('CLASSIFICATION REPORT')
print('=' * 50)
print(classification_report(y_true_classes, y_pred_classes, target_names=target_names))

## 🔥 10. Confusion Matrix

In [ ]:
conf_mat = confusion_matrix(y_true_classes, y_pred_classes)
conf_mat_norm = conf_mat.astype('float') / conf_mat.sum(axis=1)[:, np.newaxis]
conf_mat2 = np.around(conf_mat_norm, decimals=2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Oranges',
            xticklabels=target_names, yticklabels=target_names, ax=ax1)
ax1.set_title('Raw Confusion Matrix')
ax1.set_ylabel('True Label')
ax1.set_xlabel('Predicted Label')

sns.heatmap(conf_mat2, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names, ax=ax2)
ax2.set_title('Normalized Confusion Matrix')
ax2.set_ylabel('True Label')
ax2.set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

# Lưu file .dat
dat_path = os.path.join(OUTPUT_DIR, 'dat', 'conf_mat_cnn_balanced.dat')
with open(dat_path, 'wb') as f:
    for line in np.matrix(conf_mat2):
        np.savetxt(f, line, fmt='%.2f')
print(f'💾 Confusion matrix .dat lưu tại: {dat_path}')

## 💾 11. Xuất Trọng Số & Tham Số Mô Hình

In [ ]:
def save_weights_to_json(model, filename):
    """Lưu toàn bộ Weights & Biases dạng JSON."""
    weights_dict = {}
    for layer in model.layers:
        weights = layer.get_weights()
        if len(weights) > 0:
            weights_dict[f'{layer.name}_Weights'] = weights[0].tolist()
            weights_dict[f'{layer.name}_Biases'] = weights[1].tolist()
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(weights_dict, f, indent=4)
    print(f'✅ Weights JSON : {filename}')


def save_weights_to_excel(model, filename):
    """Lưu ma trận trọng số vào từng Sheet của Excel."""
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        for layer in model.layers:
            weights = layer.get_weights()
            if len(weights) > 0:
                w, b = weights[0], weights[1]
                pd.DataFrame(w.reshape(-1, w.shape[-1])).to_excel(
                    writer, sheet_name=f'{layer.name}_Weights')
                pd.DataFrame(b.reshape(-1, 1)).to_excel(
                    writer, sheet_name=f'{layer.name}_Biases')
    print(f'✅ Weights Excel: {filename}')


def save_layer_parameters_to_json(model, filename):
    """Trích xuất siêu tham số từng Layer thành JSON."""
    layers_info = []
    for i, layer in enumerate(model.layers):
        try:
            in_shape = str(layer.input_shape)
            out_shape = str(layer.output_shape)
        except AttributeError:
            in_shape = str(layer.input.shape) if hasattr(layer, 'input') else 'Unknown'
            out_shape = str(layer.output.shape) if hasattr(layer, 'output') else 'Unknown'

        info = {
            'index': i,
            'layer_name': layer.name,
            'layer_type': layer.__class__.__name__,
            'input_shape': in_shape,
            'output_shape': out_shape,
            'total_params': layer.count_params(),
            'trainable': layer.trainable
        }
        config = layer.get_config()
        for field in ['activation', 'kernel_size', 'strides', 'padding', 'pool_size', 'units', 'filters']:
            if field in config:
                if field == 'activation' and hasattr(layer, 'activation') and layer.activation:
                    info[field] = layer.activation.__name__ if hasattr(layer.activation, '__name__') else str(layer.activation)
                else:
                    info[field] = config[field]
        layers_info.append(info)

    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(layers_info, f, indent=4, ensure_ascii=False)
    print(f'✅ Params JSON  : {filename}')


def save_layer_parameters_to_excel(model, filename):
    """Lưu metadata kiến trúc các Layer vào Excel."""
    layers_summary = []
    for i, layer in enumerate(model.layers):
        config = layer.get_config()
        try:
            in_shape = str(layer.input_shape)
            out_shape = str(layer.output_shape)
        except AttributeError:
            in_shape = str(layer.input.shape) if hasattr(layer, 'input') else 'Unknown'
            out_shape = str(layer.output.shape) if hasattr(layer, 'output') else 'Unknown'

        layers_summary.append({
            'No.': i,
            'Layer Name': layer.name,
            'Layer Type': layer.__class__.__name__,
            'Input Shape': in_shape,
            'Output Shape': out_shape,
            'Filters/Units': config.get('filters', config.get('units', '-')),
            'Kernel/Pool Size': str(config.get('kernel_size', config.get('pool_size', '-'))),
            'Strides': str(config.get('strides', '-')),
            'Padding': config.get('padding', '-'),
            'Activation': layer.activation.__name__ if hasattr(layer, 'activation') and layer.activation and hasattr(layer.activation, '__name__') else '-',
            'Total Params': layer.count_params()
        })

    df = pd.DataFrame(layers_summary)
    mode = 'a' if os.path.exists(filename) else 'w'
    kwargs = {'if_sheet_exists': 'replace'} if mode == 'a' else {}
    with pd.ExcelWriter(filename, engine='openpyxl', mode=mode, **kwargs) as writer:
        df.to_excel(writer, sheet_name='Layers_Architecture', index=False)
    print(f'✅ Params Excel : {filename}')


# --- Chạy export ---
print('📤 Đang xuất trọng số và tham số...')
save_weights_to_json(malware_model, os.path.join(OUTPUT_DIR, 'malware_balanced_weights.json'))
save_weights_to_excel(malware_model, os.path.join(OUTPUT_DIR, 'malware_balanced_weights.xlsx'))
save_layer_parameters_to_json(malware_model, os.path.join(OUTPUT_DIR, 'malware_balanced_parameters.json'))
save_layer_parameters_to_excel(malware_model, os.path.join(OUTPUT_DIR, 'malware_balanced_weights.xlsx'))

print(f'\n🎉 Hoàn thành! Toàn bộ kết quả lưu tại: {OUTPUT_DIR}')